# Part 2 · The agent, streamed as Server-Sent Events

A brief takes tens of seconds. Instead of one JSON answer at the end, the service streams the run: each step of
the agent becomes an **SSE event** the UI shows as it happens. This part reads the three pieces that make that work:

| File | Role |
| --- | --- |
| [`agent/tools.py`](../app/agent/tools.py), [`agent/graph.py`](../app/agent/graph.py) | the Research Brief Agent (Deep Agent, web search, `save_brief` that waits for approval) |
| [`services/streaming.py`](../app/services/streaming.py) | LangGraph stream chunks → named events (`to_events`), one whole run as SSE (`run_events`) |
| [`routers/runs.py`](../app/routers/runs.py) | `POST /threads/{thread_id}/runs` returns that stream with `EventSourceResponse` |

Docs: [FastAPI Server-Sent Events](https://fastapi.tiangolo.com/tutorial/server-sent-events/) ·
[LangGraph streaming](https://docs.langchain.com/oss/python/langgraph/streaming) ·
[MDN: Using server-sent events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events)

In [1]:
import logging
import sys

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", stream=sys.stdout)  # log lines in the normal output
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from utils import check_server, client, short, show  # notebooks/utils.py

check_server()

server: ok


## 1. The agent

`build_agent` is called once by the lifespan. `interrupt_on` pauses the run before `save_brief`, and the tools read
the user from the run **context** (filled from the token), never from the model's arguments. `internet_search` also
reports progress with `get_stream_writer()`: that becomes the `progress` event.

> **Read in the editor:** [`app/agent/graph.py`](../app/agent/graph.py), `build_agent`, lines 12–23.

> **Read in the editor:** [`app/agent/tools.py`](../app/agent/tools.py), `internet_search`, lines 31–49.

## 2. From LangGraph chunks to events: `to_events`

`agent.astream(..., stream_mode=["updates", "messages", "custom"])` yields `(mode, chunk)` pairs. `to_events` maps
each one to zero or more `(event, data)` pairs. The `values` mode (whole state after each step) is not used: it
repeats every message each time and would expose internal state to the browser.

> **Read in the editor:** [`app/services/streaming.py`](../app/services/streaming.py), `to_events`, lines 25–48.

Run the agent **in this notebook**, with its checkpointer on the service's SQLite file (`output/agent.db`), and print, for each chunk, its mode, its raw shape and
the events it becomes. Tokens are counted, not printed: this question ends at the approval step, and the model writes reply text only after the decision (Part 3), so the count is 0 here. Middleware hooks such as `HumanInTheLoopMiddleware.after_model` give no event unless they add a message (the call limit adds an error `ToolMessage`, hence one `tool_result`).

In [2]:
from app.agent import graph
from app.services.streaming import to_events

from langchain.chat_models import init_chat_model
import uuid

import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from app.agent.tools import Context
from app.core.config import get_settings

conn = await aiosqlite.connect(get_settings().db_path)  # the same database file as the service
agent = graph.build_agent(init_chat_model(get_settings().model), AsyncSqliteSaver(conn))
config = {"configurable": {"thread_id": f"notebook-{uuid.uuid4().hex[:6]}"}}  # a new thread each run
payload = {"messages": [{"role": "user", "content": "Brief on how loud heat pumps are for neighbours."}]}

tokens = 0
async for mode, chunk in agent.astream(payload, config, context=Context(user_id="carol"),
                                       stream_mode=["updates", "messages", "custom"]):
    events = to_events(mode, chunk)
    if mode == "messages":
        tokens += len(events)
        continue
    raw = list(chunk) if mode == "updates" else chunk
    print(f"{mode:<8} {short(raw, 60):<62} -> {[name for name, _ in events]}")
print("token events:", tokens)

updates  ['PatchToolCallsMiddleware.before_agent']                      -> []
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['HumanInTheLoopMiddleware.after_model']                       -> []
updates  ['ToolCallLimitMiddleware[internet_search].after_model']       -> []
custom   {'stage': 'searching', 'query': 'air source heat pump outdoo…  -> ['progress']
INFO | internet_search('air source heat pump outdoor unit noise neighbour decibels distance g…) -> ok 2454 ms
updates  ['tools']                                                      -> ['step', 'tool_result']
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['HumanInTheLoopMiddleware.after_model']                       -> []
updates  ['ToolCallLimitMiddleware[internet_search].after_model']       -> []
custom   {'stage': 'searching', 'query': 'site:energysavingtrust.org.…  -> ['progress']
INFO | internet_search('site:en

## 3. A whole run: `run_events`

`run_events` wraps the loop: it sends `metadata` first, the mapped events, then `done` (or `error`); it gives every
event an `id`, enforces one run per thread and a time limit for the whole run. It takes plain objects (agent, the
registry of running threads, user, thread), not the HTTP request, so it can be called here directly.

> **Read in the editor:** [`app/services/streaming.py`](../app/services/streaming.py), `run_events`, lines 51–98.

FastAPI turns each `ServerSentEvent` into text lines. `format_sse_event` is the function it uses; the cell runs a
new question through `run_events` and prints the first events **exactly as they go on the wire**, then stops reading.

In [3]:
from app.services.streaming import run_events

import json

from fastapi.sse import format_sse_event

payload = {"messages": [{"role": "user", "content": "Brief on induction cooktops vs gas."}]}
stream = run_events(agent, {}, "carol", f"notebook-{uuid.uuid4().hex[:6]}", payload)
for _ in range(4):
    e = await anext(stream)
    print(format_sse_event(data_str=json.dumps(e.data), event=e.event, id=e.id).decode(), end="")
await stream.aclose()  # stop the run: no more model or tool calls

INFO | run 9a56ae3e started: user=carol thread=notebook-d7a833
event: metadata
data: {"run_id": "9a56ae3e", "thread_id": "notebook-d7a833"}
id: 9a56ae3e-1

event: step
data: {"node": "model"}
id: 9a56ae3e-2

event: tool_call
data: {"id": "call_9NpD8fi6ZAMzl69Dym2ws8MB", "name": "internet_search", "args": {"query": "induction cooktops versus gas efficiency safety emissions studies official energy consumer research"}}
id: 9a56ae3e-3

event: progress
data: {"stage": "searching", "query": "induction cooktops versus gas efficiency safety emissions studies official energy consumer research"}
id: 9a56ae3e-4



## 4. The route and the real HTTP stream

`create_run` declares `response_class=EventSourceResponse` and yields the events of `run_events`. Its parameters are
the checks of Part 3 (`IdleThread`, `CurrentUser`): they run **before** the first byte is sent.

> **Read in the editor:** [`app/routers/runs.py`](../app/routers/runs.py), `create_run`, lines 18–27.

Now the running service: sign in as Carol (`carol` / `carol-pass-2026`, created by `utils.client`), create a thread, send a message, print the response headers and every event in
the order it arrives. The run pauses before `save_brief`: the stream sends `interrupt`, then `done` with status
`interrupted`, and closes (Part 3 resumes it).

In [4]:
from utils import sse_events

carol = client("carol")  # Part 2 uses its own account; Part 3 creates Alice and Bob
thread_id = carol.post("/threads").json()["thread_id"]
events = []
with carol.stream("POST", f"/threads/{thread_id}/runs",
                  json={"message": "Brief on how well air-source heat pumps work below 0°C."}) as r:
    print("status:", r.status_code, "| content-type:", r.headers["content-type"], "| cache-control:",
          r.headers["cache-control"], "\n")
    for e in sse_events(r):
        events.append(e)
        show(e["event"], e["data"])

status: 200 | content-type: text/event-stream; charset=utf-8 | cache-control: no-cache 

[metadata] run 8cfdc740 on thread 8b933ebc1e57
[step] model
[tool_call] internet_search({"query": "air source heat pump performance below 0°C cold weather COP field study defrost…)
[progress] searching: air source heat pump performance below 0°C cold weather COP field study defrost
[step] tools
[tool_result] internet_search -> success: [{"title": "Weather in global", "url": "https://www.weatherapi.com/", "content":…
[step] model
[tool_call] internet_search({"query": "site:energy.gov air source heat pumps cold climates below 0°F efficiency perfor…)
[progress] searching: site:energy.gov air source heat pumps cold climates below 0°F efficiency performance cold weather
[step] tools
[tool_result] internet_search -> success: [{"title": "Field Performance of Inverter-Driven Heat Pumps in Cold ...", "url":…
[step] model
[tool_call] internet_search({"query": "NREL air source heat pump cold temperature COP -

In [5]:
from collections import Counter

print("events:", dict(Counter(e["event"] for e in events)))
action = next(e["data"] for e in events if e["event"] == "interrupt")["action_requests"][0]
print("waiting for approval:", action["name"], "| title:", action["args"]["title"])
print("thread status:", carol.get(f"/threads/{thread_id}").json()["status"])

events: {'metadata': 1, 'step': 7, 'tool_call': 4, 'progress': 3, 'tool_result': 3, 'interrupt': 1, 'done': 1}
waiting for approval: save_brief | title: Air-source heat pumps below 0°C
thread status: interrupted
